In [ ]:
import os
os.environ["CBEAM_BACKEND"] = "numpy"

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# =====================================================================
# CELL 2: IMPORT BATCH PIPELINE
# =====================================================================

from batch_propagation_pipeline import *
ideal_grid_positions = [
    (0.0000, 0.0000), (1.0000, 0.0000), (0.5000, 0.8660), (-0.5000, 0.8660),
    (-1.0000, 0.0000), (-0.5000, -0.8660), (0.5000, -0.8660), (2.0000, 0.0000),
    (1.5000, 0.8660), (1.0000, 1.7321), (0.0000, 1.7321), (-1.0000, 1.7321),
    (-1.5000, 0.8660), (-2.0000, 0.0000), (-1.5000, -0.8660), (-1.0000, -1.7321),
    (0.0000, -1.7321), (1.0000, -1.7321), (1.5000, -0.8660)
]
print("Batch pipeline imported successfully")

In [ ]:

import time
import numpy as np
 
from batch_propagation_pipeline import (
    get_waveguide_properties,
    IncidentFieldGenerator,
    ModalProjector,
    BatchPropagationPipeline,
    DEFAULT_GRID_RESOLUTION,
)
 
 
def profile_pipeline_preset(prop12, p, ifunc, grid_resolution=DEFAULT_GRID_RESOLUTION):
    """
    Manually replays BatchPropagationPipeline.__init__'s steps for one
    wavelength, timing each one. Does NOT construct a real pipeline --
    this is read-only profiling against an already-built prop12.
    """
    timings = {}
 
    t0 = time.perf_counter()
    wvg_props_input = get_waveguide_properties(prop12, mesh_z=0)
    t1 = time.perf_counter()
    timings["get_waveguide_properties(input, mesh_z=0)"] = t1 - t0
 
    wvg_props_output = get_waveguide_properties(prop12, mesh_z=p["z_ex"])
    t2 = time.perf_counter()
    timings["get_waveguide_properties(output, mesh_z=z_ex)"] = t2 - t1
 
    field_gen = IncidentFieldGenerator(p, ifunc, xp=np)
    t3 = time.perf_counter()
    timings["IncidentFieldGenerator.__init__ (mask + ifunc_matrix)"] = t3 - t2
 
    field_gen.precompute_interpolation_weights(wvg_props_input["points"])
    t4 = time.perf_counter()
    timings["precompute_interpolation_weights"] = t4 - t3
 
    modal_proj = ModalProjector(wvg_props_input, xp=np)
    t5 = time.perf_counter()
    timings["ModalProjector.__init__"] = t5 - t4
 
    # Replicates _precompute_delaunay_grid without needing a fully
    # constructed BatchPropagationPipeline instance.
    dummy = BatchPropagationPipeline.__new__(BatchPropagationPipeline)
    dummy.wvg_props_output = wvg_props_output
    _ = dummy._precompute_delaunay_grid(grid_resolution)
    t6 = time.perf_counter()
    timings[f"_precompute_delaunay_grid(resolution={grid_resolution})"] = t6 - t5
 
    print("\n" + "=" * 60)
    print("PRESET-PHASE TIMING BREAKDOWN")
    print("=" * 60)
    total = t6 - t0
    for label, dt in timings.items():
        bar = "#" * max(1, int(40 * dt / max(total, 1e-9)))
        print(f"{dt:7.3f} s  {bar:<40s}  {label}")
    print("-" * 60)
    print(f"{total:7.3f} s  TOTAL")
    print("=" * 60)
 
    return timings

In [ ]:
# =====================================================================
# CELL 3: INITIALIZE SYSTEM PARAMETERS
# =====================================================================

p = get_simulation_parameters(3, 0.81)

print("\n" + "="*60)
print("SIMULATION PARAMETERS")
print("="*60)
print(f"Wavelength: {p['wl']} μm")
print(f"Lantern length: {p['z_ex']} μm")
print(f"Cladding radius: {p['rclad']} μm")
print(f"Jacket radius: {p['rjack']} μm")
print(f"Core radius: {p['rcore']:.3f} μm")
print(f"Taper factor: {p['taper_factor']}")
print(f"Available influence function: {p['ifunc_file']}")

In [ ]:
# =====================================================================
# CELL 4: LOAD INFLUENCE FUNCTION AND PROPAGATOR
# =====================================================================

import specula

print("Initializing SPECULA GPU...")
specula.init(0)
from specula.data_objects.ifunc import IFunc

print("Loading influence function...")
# UPDATE THIS LINE TO EXPLICITLY PASS THE DEVICE INDEX:
print(p["ifunc_file"])
ifunc = IFunc.restore(p["ifunc_file"]) 
mask = ifunc.mask_inf_func.get()



# Find the bounding box or centroid
y_cent = np.mean(np.where(mask > 0)[0])
x_cent = np.mean(np.where(mask > 0)[1])
# Desired center (assuming square mask)
center_y = mask.shape[0] // 2
center_x = mask.shape[1] // 2
shift_y = int(center_y - y_cent)
shift_x = int(center_x - x_cent)

print('shift_x, shift_y', shift_x, shift_y)

mask_centered = np.roll(mask, shift_y, axis=0)
mask_centered = np.roll(mask_centered, shift_x, axis=1)
ifunc.mask_inf_func.set(mask_centered)

# Replace the mask
ifunc.mask_inf_func.set(mask_centered)

print(f"  Influence function modes: {len(ifunc.influence_function)}")

print("\nBuilding photonic lantern propagator...")
prop12 = build_and_characterize_lantern(p)

print(f"  Propagator type: {type(prop12).__name__}")
print(f"  Total z range: {prop12.zs[0]:.0f} to {prop12.zs[-1]:.0f} μm")

In [ ]:
# =====================================================================
# CELL 5: CREATE BATCH PROPAGATION PIPELINE
# =====================================================================

print("Creating batch propagation pipeline...\n")
pipeline = BatchPropagationPipeline(prop12, p, ifunc)

diagnose_input_psf(pipeline)

print("Waveguide Properties:")
print(f"  Input modes: {pipeline.wvg_props_input['n_modes']}")
print(f"  Output modes: {pipeline.wvg_props_output['n_modes']}")
print(f"  Mesh points at input: {pipeline.wvg_props_input['points'].shape[0]}")
print(f"  Mesh points at output: {pipeline.wvg_props_output['points'].shape[0]}")

In [ ]:
profile_pipeline_preset(prop12, p, ifunc)

In [ ]:
#coeff1, _labels1 = create_random_aberration_configs(n=1, m=10, minv=-0, maxv=0)
coeff0, _labels0 = create_random_aberration_configs(n=100, m=9, minv=400, maxv=400)
coeff1, labels1 = create_ramp_aberration_configs([0],20,0,400)

In [ ]:
print("\nGenerating batch of input modal coefficients...\n")
import time
print(coeff0.shape)
start_time = time.time()
u0_batch = pipeline.generate_batch_modal_coefficients(coeff0, use_gpu=False)
elapsed = time.time() - start_time
print(f"\generate_batch_modal_coefficients Complete:")
print(f"  Elapsed time: {elapsed:.1f} seconds")
print(f"Input Modal Coefficient Batch:")
print(f"  Shape: {u0_batch.shape}")
print(f"  Dtype: {u0_batch.dtype}")
print(f"  Memory: {u0_batch.nbytes / 1e6:.1f} MB")

u1_batch = pipeline.generate_batch_modal_coefficients(coeff1)


In [ ]:
# =====================================================================
# CELL 8: PROPAGATE BATCH THROUGH LANTERN
# =====================================================================
import time
start_time = time.time()

print("\nPropagating batch through lantern...")
#uf_batch, zs, us_batch = pipeline.propagate_batch_single(u1_batch)
#uf_batch, zs, us_batch = pipeline.propagate_batch(u1_batch)

elapsed = time.time() - start_time

print(f"\nPropagation Complete:")
print(f"  Elapsed time: {elapsed:.1f} seconds")
start_time = time.time()


#uf_batch, zs, us_batch = pipeline.propagate_batch_single(u0_batch)
uf_batch0, zs0, us_batch0 = pipeline.propagate_batch(u0_batch)
uf_batch1, zs1, us_batch1 = pipeline.propagate_batch(u1_batch)

elapsed = time.time() - start_time

print(f"\nPropagation Complete:")
print(f"  Elapsed time: {elapsed:.1f} seconds")
print(f"  Output modal coefficients shape: {uf_batch0.shape}")
print(f"  Propagation z-points: {len(zs0)}")

In [ ]:
uf_batch0[0]

In [ ]:
# =====================================================================
# CELL 9: RECONSTRUCT OUTPUT SPATIAL FIELDS
# =====================================================================

print("Reconstructing output spatial fields...\n")

E_output_batch0 = pipeline.reconstruct_batch_output_fields(uf_batch0)
E_output_batch1 = pipeline.reconstruct_batch_output_fields(uf_batch1)


print(f"Output Spatial Fields:")
print(f"  Shape: {E_output_batch0.shape}")
print(f"  Dtype: {E_output_batch0.dtype}")
print(f"  Memory: {E_output_batch0.nbytes / 1e6:.1f} MB")


In [ ]:
# =====================================================================
# CELL 10: INTERPOLATE TO REGULAR GRID
# =====================================================================

print("Interpolating output fields to regular grid...\n")

uf_2d_batch0, X_plot0, Y_plot0 = pipeline.interpolate_output_to_grid(E_output_batch0, grid_resolution=200)
uf_2d_batch1, X_plot1, Y_plot1 = pipeline.interpolate_output_to_grid(E_output_batch1, grid_resolution=200)

print(f"Interpolated Intensity Maps:")
print(f"  Shape: {uf_2d_batch0.shape}")
print(f"  X range: [{X_plot0.min():.2f}, {X_plot0.max():.2f}] μm")
print(f"  Y range: [{Y_plot0.min():.2f}, {Y_plot0.max():.2f}] μm")
print(f"  Memory: {uf_2d_batch0.nbytes / 1e6:.1f} MB")

In [ ]:
# =====================================================================
# CELL 11: VISUALIZE BATCH OUTPUT
# =====================================================================

print("Generating visualization...\n")

# Create titles for each field
titles = [] #[f"Mode {c['mode_idx']}, {c['amplitude_nm']:.0f} nm" 
          #for c in aberration_configs]

# Visualize
visualize_batch_output(uf_2d_batch1[0:2], X_plot1, Y_plot1, titles, maxv=1, show_arrow=False)

In [ ]:
# --- Step 1: Detect core centers from the first output field ---
first_intensity = uf_2d_batch0.sum(axis=0)
# shape (grid_resolution, grid_resolution)
# But we need the spatial coordinates X_plot, Y_plot from the pipeline.
# Actually, we have X_plot, Y_plot from the interpolation.

# Reuse your calibration function, but now pass the 2D intensity map and the mesh points?
# The calibration function expects a (N_points,) profile and mesh.points.
# But we already have the interpolated grid. Let's create a function that works directly on grid.


# Detect centers from the first output field
core_centers = detect_centers_from_grid(first_intensity, X_plot0, Y_plot0)
print(f"Detected {len(core_centers)} core centers")

In [ ]:

# Detect centers once
x_min, y_min = X_plot0[0,0], Y_plot0[0,0]
dx = X_plot0[0,1] - X_plot0[0,0]
dy = Y_plot0[1,0] - Y_plot0[0,0]

core_centers = detect_centers_from_grid(first_intensity, X_plot0, Y_plot0)
ideal_permutation = map_evaluated_to_ideal_geometry(core_centers, ideal_grid_positions)


In [ ]:

for i in range(len(uf_2d_batch1)):
    # Extract signals for field i
    signals = collect_subpixel_signals(uf_2d_batch1[i], x_min, y_min, dx, dy, core_centers)
    standardized = np.zeros(19)
    standardized[ideal_permutation] = signals*10
    display_hex_grid_plots(ideal_grid_positions, standardized)